# MM_5M on Google Colab

<a href="https://colab.research.google.com/github/arborrhythms/BasicModel/blob/main/data/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook clones `arborrhythms/BasicModel` from GitHub, installs dependencies, creates a Colab runtime copy of `data/MM_5M.xml`, and offers two paths:

- `train`: run a bounded MM_5M training job through `bin/train.py`.
- `run`: load MM_5M and run an inference-style forward pass on a prompt.

The default settings use the real MM_5M architecture with synthetic inline text, `batchSize=1`, no checkpoint autoload, no autosave, and one training batch. That keeps the first run practical on a free Colab GPU. For FineWeb-backed training, set `DATASET = "text"`; that will download a FineWeb shard.

In [1]:
#@title Run settings
REPO_URL = "https://github.com/arborrhythms/BasicModel.git"
BRANCH = "main"
PROJECT_DIR = "/content/BasicModel"

# Choose "train", "run", or "train_then_run".
ACTION = "train"  #@param ["train", "run", "train_then_run"]

# Use "inline" for a no-dataset smoke run, or "text" for FineWeb shards.
DATASET = "inline"  #@param ["inline", "text"]

# Colab-safe training defaults. Increase these for a real training run.
BATCH_SIZE = 1  #@param {type:"integer"}
NUM_EPOCHS = 1  #@param {type:"integer"}
BATCHES = 1  #@param {type:"integer"}
RUN_TEST = False  #@param {type:"boolean"}

# Checkpoint behavior. SAVE_CHECKPOINT writes data/MM_5M.ckpt in Colab.
TRAIN_AUTOLOAD = False  #@param {type:"boolean"}
SAVE_CHECKPOINT = False  #@param {type:"boolean"}
RUN_AUTOLOAD = False  #@param {type:"boolean"}

# Used only when DATASET == "text".
NUM_SHARDS = 1  #@param {type:"integer"}
MAX_DOCS = 64  #@param {type:"integer"}
RANDOM_SHARDS = False  #@param {type:"boolean"}

# Startup-oriented defaults. Set COMPILE_BACKEND="auto" for longer GPU runs.
COMPILE_BACKEND = "none"  #@param ["none", "auto", "inductor", "eager", "aot_eager"]
COMPILE_MODE = "default"  #@param ["default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs"]
AMP = "auto"  #@param ["auto", "off", "fp16", "bf16"]

# Used by the run path.
RUN_PROMPT = "hello world"  #@param {type:"string"}


In [2]:
import os
import subprocess
import sys
from pathlib import Path

def run_cmd(cmd, *, cwd=None, env=None, check=True):
    print("+", " ".join(str(x) for x in cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

project = Path(PROJECT_DIR)
if not (project / ".git").exists():
    clone_cmd = ["git", "clone", "--depth", "1"]
    if BRANCH:
        clone_cmd += ["--branch", BRANCH]
    clone_cmd += [REPO_URL, str(project)]
    run_cmd(clone_cmd)
else:
    run_cmd(["git", "-C", str(project), "pull", "--ff-only"], check=False)

os.chdir(project)
print("Project:", Path.cwd())

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run_cmd([sys.executable, "-m", "pip", "install", "-q", "requests", "pyarrow"])


+ git clone --depth 1 --branch main https://github.com/arborrhythms/BasicModel.git /content/BasicModel
Project: /content/BasicModel
+ /usr/bin/python3 -m pip install -q -r requirements.txt
+ /usr/bin/python3 -m pip install -q requests pyarrow


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'requests', 'pyarrow'], returncode=0)

In [3]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda:", torch.version.cuda)
    print("gpu:", torch.cuda.get_device_name(0))
    run_cmd(["nvidia-smi"], check=False)

def make_env():
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["MODEL_COMPILE"] = COMPILE_BACKEND
    env["MODEL_COMPILE_MODE"] = COMPILE_MODE
    env["BASICMODEL_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
    env["MODEL_AMP"] = ("fp16" if torch.cuda.is_available() else "off") if AMP == "auto" else AMP
    mpl_dir = Path("output/.mplconfig").resolve()
    mpl_dir.mkdir(parents=True, exist_ok=True)
    env["MPLCONFIGDIR"] = str(mpl_dir)
    return env

env = make_env()
print("Device:", env["BASICMODEL_DEVICE"])
print("AMP:", env["MODEL_AMP"])
print("MODEL_COMPILE:", env["MODEL_COMPILE"])


torch: 2.11.0+cu128
cuda available: True
cuda: 12.8
gpu: Tesla T4
+ nvidia-smi
Device: cuda
AMP: fp16
MODEL_COMPILE: none


In [4]:
import xml.etree.ElementTree as ET

SOURCE_XML = Path("data/MM_5M.xml")
RUNTIME_XML = Path("data/MM_5M_colab_runtime.xml")

def write_runtime_xml(*, autoload=False, autosave=False):
    tree = ET.parse(SOURCE_XML)
    root = tree.getroot()

    def ensure(path):
        node = root
        for part in path.split("/"):
            child = node.find(part)
            if child is None:
                child = ET.SubElement(node, part)
            node = child
        return node

    updates = {
        "architecture/training/batchSize": BATCH_SIZE,
        "architecture/training/numEpochs": NUM_EPOCHS,
        "architecture/training/numWorkers": 0,
        "architecture/training/autoload": autoload,
        "architecture/training/autosave": autosave,
        "architecture/training/checkpointEveryBatches": 0,
    }
    for path, value in updates.items():
        ensure(path).text = str(value).lower() if isinstance(value, bool) else str(value)

    tree.write(RUNTIME_XML, encoding="utf-8", xml_declaration=True)
    print(f"Wrote {RUNTIME_XML}")
    for path, value in updates.items():
        print(f"  {path} = {value}")
    return RUNTIME_XML


In [5]:
# Training option: invokes bin/train.py, which is the project training entry point.
if ACTION in ("train", "train_then_run"):
    train_xml = write_runtime_xml(autoload=TRAIN_AUTOLOAD, autosave=SAVE_CHECKPOINT)
    cmd = [
        sys.executable,
        "bin/train.py",
        "--model", str(train_xml),
        "--data", DATASET,
        "--num-epochs", str(NUM_EPOCHS),
        "--batches", str(BATCHES),
    ]
    if DATASET == "text":
        cmd += ["--max-docs", str(MAX_DOCS), "--num-shards", str(NUM_SHARDS)]
        if RANDOM_SHARDS:
            cmd += ["--random-shards"]
    if RUN_TEST:
        cmd += ["--test", "1"]
    if COMPILE_BACKEND != "none":
        cmd += ["--compile-mode", COMPILE_MODE]

    print("Training command:", " ".join(cmd))
    subprocess.run(cmd, cwd=Path.cwd(), env=make_env(), check=True)
else:
    print(f"Skipping training because ACTION={ACTION!r}")


Wrote data/MM_5M_colab_runtime.xml
  architecture/training/batchSize = 1
  architecture/training/numEpochs = 1
  architecture/training/numWorkers = 0
  architecture/training/autoload = False
  architecture/training/autosave = False
  architecture/training/checkpointEveryBatches = 0
Training command: /usr/bin/python3 bin/train.py --model data/MM_5M_colab_runtime.xml --data inline --num-epochs 1 --batches 1


CalledProcessError: Command '['/usr/bin/python3', 'bin/train.py', '--model', 'data/MM_5M_colab_runtime.xml', '--data', 'inline', '--num-epochs', '1', '--batches', '1']' returned non-zero exit status 1.

In [ ]:
# Running option: load MM_5M and run an inference-style forward pass on RUN_PROMPT.
if ACTION in ("run", "train_then_run"):
    run_xml = write_runtime_xml(autoload=RUN_AUTOLOAD, autosave=False)

    run_env = make_env()
    os.environ.update({
        "BASICMODEL_DEVICE": run_env["BASICMODEL_DEVICE"],
        "MODEL_COMPILE": run_env["MODEL_COMPILE"],
        "MODEL_COMPILE_MODE": run_env["MODEL_COMPILE_MODE"],
        "MODEL_AMP": run_env["MODEL_AMP"],
        "MPLCONFIGDIR": run_env["MPLCONFIGDIR"],
    })

    sys.path.insert(0, str(Path("bin").resolve()))
    from util import init_compile_backend, init_compile_mode, init_device, init_model_amp, TheDevice
    init_device()
    init_compile_backend()
    init_compile_mode()
    init_model_amp()

    from data import TheData
    from Models import BaseModel

    if DATASET == "text":
        TheData.load(DATASET, num_shards=NUM_SHARDS, max_docs=MAX_DOCS, shard_dir="data/fineweb")
    else:
        TheData.load("inline", dat={})

    model, _cfg = BaseModel.from_config(str(run_xml), data=TheData)
    model.eval()
    if COMPILE_BACKEND != "none":
        model.enable_compiled_step()

    print("Loaded model on", TheDevice)
    print("Prompt:", RUN_PROMPT)
    result = model.generate_sentence(RUN_PROMPT)
    print("Masked-position predictions:")
    if not result:
        print("  No masked positions were selected on this pass. Re-run this cell or raise maskRate in the XML.")
    else:
        for slot, original, predicted in result:
            print(f"  slot={slot} original={original!r} predicted={predicted!r}")
else:
    print(f"Skipping run because ACTION={ACTION!r}")


In [ ]:
out = Path("output")
if out.exists():
    print("Recent output files:")
    files = sorted([p for p in out.rglob("*") if p.is_file()], key=lambda p: p.stat().st_mtime, reverse=True)
    for p in files[:20]:
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  {p} ({size_mb:.2f} MB)")
else:
    print("No output directory yet.")

ckpt = Path("data/MM_5M.ckpt")
print("Checkpoint:", ckpt, "exists" if ckpt.exists() else "not present")


## Scaling Up

For a real data run, set `DATASET = "text"`, raise `MAX_DOCS`, and raise `BATCHES`. Keep `BATCH_SIZE = 1` until the run is stable on the selected Colab GPU, then increase it carefully. Set `COMPILE_BACKEND = "auto"` only for longer runs where the compile warmup cost is worth paying.

To run from a checkpoint produced in this Colab session, set `SAVE_CHECKPOINT = True` before training and `RUN_AUTOLOAD = True` before running. The checkpoint path is `data/MM_5M.ckpt` because `MM_5M_colab_runtime.xml` lives in `data/`.